# Eye-aspect-ratio (EAR) branch

Decides whether the eyes are open or closed from the **geometry of the eye landmarks**, then measures PERCLOS, eye-closure time, blink rate, a drowsiness score and a **confidence** that drops whenever the face or eyes are not clearly visible.

```
frame -> face box (YuNet, Haar as fallback) -> 68 landmarks (dlib) -> 6 points per eye -> EAR -> open / closed
      -> rolling 30 s window -> PERCLOS, closure time, blinks, score, confidence
```
The code lives in the package (`drowsiness/video/ear_branch.py`, `drowsiness/common/windowing.py`), the same code the fusion uses. This notebook goes in order: **1. one image, 2. a video file, 3. the live camera**.
From the terminal: `python -m drowsiness.video.ear_branch --source 0` (camera) or `--source data/test_vid_1.mov`.

## 0. Setup

In [1]:
# Run once in the project root, then pick the .venv kernel for this notebook:
# python3 -m venv .venv && .venv/bin/pip install -r requirements.txt
import os
import subprocess
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np

from drowsiness.common.windowing import (
    BLINK_MAX_DURATION,
    DROWSY_CLOSURE_DURATION,
    MICROSLEEP_DURATION,
    MIN_CLOSURE_DURATION,
    calculate_drowsiness_score,
    channel_output,
    closure_level,
    find_closures,
)
from drowsiness.video.ear_branch import DEFAULT_THRESHOLD, EarBranch, calculate_ear, calibrate_threshold
from drowsiness.video.runner import run_video, summary

IMAGE_PATH = "data/test2.jpg"  # your own test image (not in git)
VIDEO_PATH = "data/test_vid_1.mov"  # your own test video (not in git)

# dlib's 68-point landmark model and the YuNet face detector; download_models.py fetches them if missing.
if not (os.path.exists("shape_predictor_68_face_landmarks.dat") and os.path.exists("models/face_detection_yunet_2023mar.onnx")):
    subprocess.run([sys.executable, "download_models.py"], check=True)

## 1. Step 1: a single image
**Face -> landmarks -> eye points -> EAR.** The branch is created with `calibrate_s=0` (no waiting for calibration, as this is one picture).

In [2]:
branch = EarBranch(calibrate_s=0)
branch.start()

image = cv2.imread(IMAGE_PATH)
if image is None:
    raise FileNotFoundError(f"Could not read image: {IMAGE_PATH}")
print("image:", image.shape)

box, landmarks = branch.locate(image)  # the largest face, and its 68 landmark points
if box is None:
    raise RuntimeError("No face found in the image")
print("face box (x, y, w, h):", box, "  landmarks:", landmarks.shape)

image: (960, 1440, 3)
face box (x, y, w, h): (541, 230, 480, 631)   landmarks: (68, 2)


In [3]:
shown = image.copy()
x, y, w, h = box
cv2.rectangle(shown, (x, y), (x + w, y + h), (0, 255, 0), 2)
for i, (px, py) in enumerate(landmarks):
    cv2.circle(shown, (int(px), int(py)), 2, (0, 255, 0), -1)
    cv2.putText(shown, str(i), (int(px) + 3, int(py) - 3), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (255, 0, 0), 1)
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(shown, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

Points 36-41 are the **left eye** and 42-47 the **right eye**, six points each: two corners (0 and 3), two on the upper lid (1, 2), two on the lower lid (5, 4).
The **eye aspect ratio** is the two vertical gaps divided by twice the horizontal width, so it is about 0.3 for an open eye and falls towards 0 as the eye closes, whatever the size of the face in the picture:

`EAR = ( |p1 - p5| + |p2 - p4| ) / ( 2 x |p0 - p3| )`

In [4]:
left_eye, right_eye = landmarks[36:42], landmarks[42:48]
left_ear, right_ear = calculate_ear(left_eye), calculate_ear(right_eye)
print(f"Left EAR : {left_ear:.3f}\nRight EAR: {right_ear:.3f}\nAverage EAR: {(left_ear + right_ear) / 2:.3f}")

eye_image = image.copy()
for px, py in np.vstack([left_eye, right_eye]):
    cv2.circle(eye_image, (int(px), int(py)), 4, (0, 255, 0), -1)
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(eye_image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

Left EAR : 0.390
Right EAR: 0.400
Average EAR: 0.395


### The threshold: adaptive, not fixed
A fixed "closed below 0.20" is too strict for some people and too loose for others (open-eye EAR is about 0.22 for small eyes and 0.35 for wide eyes). So the threshold is set to **0.7 x the person's own median open-eye EAR** (limited to 0.12-0.30), and falls back to 0.20 if too few good frames are seen. This test image shows open eyes, so it gives this person's threshold.

### Confidence of one frame
**Confidence** (0-1) says how much to trust the EAR of this frame. It is the *minimum* of seven simple factors (a chain is only as strong as its weakest link), so any one problem lowers it:

| factor | drops when |
|---|---|
| `size` | eyes are only a few pixels wide (far away / low resolution) |
| `pose` | head is turned: one eye looks narrower than the other |
| `inside` | eye points fall outside the image (face cut off) |
| `light` | eye region is too dark or over-exposed |
| `stable` | eye landmarks jump between frames (tracking is jittery) |
| `agree` | the two eyes give very different EAR (one eye hidden, wrong landmarks) |
| `inbox` | the eye points are not in the upper part of the face box (false face detection, bad landmark fit) |

No face gives confidence 0. Frames below `MIN_CONF` are labelled `UNKNOWN`, so they are never counted as "open" or "closed".

In [5]:
res = branch.analyse(image)
branch.threshold = calibrate_threshold([res["ear"]], DEFAULT_THRESHOLD, min_samples=1)  # this person's threshold
res = branch.analyse(image)

print(f"Adaptive EAR threshold: {branch.threshold:.3f}")
print(f"EAR {res['ear']:.3f}  state {res['state']}  confidence {res['confidence']:.2f}")
print({k: round(v, 2) for k, v in res["factors"].items()})

Adaptive EAR threshold: 0.277
EAR 0.395  state OPEN  confidence 1.00
{'size': 1.0, 'pose': 1.0, 'inside': 1.0, 'light': 1.0, 'stable': 1.0, 'agree': 1.0, 'inbox': 1.0}


## 2. Step 2: a video file
`run_video` feeds every frame to the branch (the file keeps its own timestamps, so the timing is right however fast it is processed). It never skips a frame, also those without a face, so time stays correct.
The threshold found in step 1 is used from the first frame (`calibrate_s=0`).

In [6]:
video_branch = EarBranch(window_s=30, calibrate_s=0, threshold=branch.threshold)
video = run_video(video_branch, VIDEO_PATH, show=None, save_to="runs/ear_video_annotated.mp4", log_csv="runs/ear_video_log.csv")

time_history, ear_history = video["t"], video["ear"]
eye_state_history, conf_history = video["state"], video["confidence"]
fps = len(time_history) / time_history[-1]
print(f"Video: {len(time_history)} frames, {time_history[-1]:.1f} s, {fps:.0f} FPS in the file")
print("Speed:", summary(video))
print("Frames with a usable face:", sum(s != "UNKNOWN" for s in eye_state_history))
print("annotated video: runs/ear_video_annotated.mp4   per-frame log: runs/ear_video_log.csv")

Stream ended or camera stopped delivering frames.
Video: 641 frames, 21.6 s, 30 FPS in the file
Speed: 641 frames in 21.6 s; processing 7.4 ms per frame (= 135 FPS possible, 95th percentile 7.7 ms)
Frames with a usable face: 587
annotated video: runs/ear_video_annotated.mp4   per-frame log: runs/ear_video_log.csv


In [7]:
plt.figure(figsize=(15, 5))
plt.plot(time_history, ear_history)
plt.axhline(branch.threshold, linestyle="--", label="EAR threshold")
plt.xlabel("Time (seconds)")
plt.ylabel("EAR")
plt.title("Eye Aspect Ratio over Time")
plt.legend()
plt.grid()
plt.show()

### Eye closures and their levels
A **closure** is a stretch of consecutive CLOSED frames; its length comes from the timestamps. Short tracking losses (up to 0.5 s) between closed frames are bridged, since eyes tend to be lost exactly while they are closed. The levels (literature values, with 1 s and 2 s as engineering choices inside the microsleep range; sources in `drowsiness/common/windowing.py`):

| closure length | level |
|---|---|
| up to 0.4 s | BLINK |
| 0.4 to 1.0 s | SLOW BLINK |
| 1.0 s or more | DROWSY |
| 2.0 s or more | MICROSLEEP |

In [8]:
closures = [c for c in find_closures(time_history, eye_state_history) if c["duration"] >= MIN_CLOSURE_DURATION]
print("Number of eye closures:", len(closures))
for i, closure in enumerate(closures):
    print(f"Closure {i + 1}: {closure['duration']:.3f} seconds  ({closure_level(closure['duration'])})")

blinks = [c for c in closures if c["duration"] <= BLINK_MAX_DURATION]
slow_blinks = [c for c in closures if BLINK_MAX_DURATION < c["duration"] < DROWSY_CLOSURE_DURATION]
prolonged_closures = [c for c in closures if c["duration"] >= DROWSY_CLOSURE_DURATION]
microsleeps = [c for c in closures if c["duration"] >= MICROSLEEP_DURATION]
print(
    f"\nBlinks: {len(blinks)}   Slow blinks: {len(slow_blinks)}   Prolonged closures: {len(prolonged_closures)}   Microsleeps: {len(microsleeps)}"
)

video_duration = time_history[-1]
blink_frequency = len(blinks) / video_duration * 60
print(f"Blink frequency: {blink_frequency:.2f} blinks/minute")

Number of eye closures: 12
Closure 1: 0.267 seconds  (BLINK)
Closure 2: 0.200 seconds  (BLINK)
Closure 3: 0.133 seconds  (BLINK)
Closure 4: 0.233 seconds  (BLINK)
Closure 5: 0.300 seconds  (BLINK)
Closure 6: 0.133 seconds  (BLINK)
Closure 7: 3.267 seconds  (MICROSLEEP)
Closure 8: 0.367 seconds  (BLINK)
Closure 9: 0.233 seconds  (BLINK)
Closure 10: 0.167 seconds  (BLINK)
Closure 11: 0.200 seconds  (BLINK)
Closure 12: 0.867 seconds  (SLOW BLINK)

Blinks: 10   Slow blinks: 1   Prolonged closures: 1   Microsleeps: 1
Blink frequency: 27.82 blinks/minute


### PERCLOS and the drowsiness state
**PERCLOS** is the share of time the eyes are closed. It counts only frames whose state is known: a lost face is not "alert".

In [9]:
closed_frames = sum(state == "CLOSED" for state in eye_state_history)
total_frames = sum(state != "UNKNOWN" for state in eye_state_history)
perclos = closed_frames / total_frames if total_frames else float("nan")
print(f"PERCLOS: {perclos:.3f}  ({perclos * 100:.2f}%)")


def detect_drowsiness(perclos, prolonged_closures):
    if any(c["duration"] >= MICROSLEEP_DURATION for c in prolonged_closures):
        return "MICROSLEEP"
    if prolonged_closures or perclos > 0.30:
        return "DROWSY"
    return "ALERT"


print("Final state:", detect_drowsiness(perclos, prolonged_closures))

longest_closure = max([c["duration"] for c in closures], default=0.0)
drowsiness_score = calculate_drowsiness_score(perclos, longest_closure, blink_frequency, video_duration)
print(
    f"Drowsiness score: {drowsiness_score:.2f}   ->  ",
    "DROWSY" if drowsiness_score >= 0.7 else "WARNING" if drowsiness_score >= 0.4 else "ALERT",
)

PERCLOS: 0.327  (32.71%)
Final state: MICROSLEEP
Drowsiness score: 0.85   ->   DROWSY


In [10]:
plt.figure(figsize=(15, 5))
plt.plot(time_history, ear_history, label="EAR")
plt.axhline(branch.threshold, linestyle="--", label="Threshold")
for closure in prolonged_closures:
    plt.axvspan(closure["start_time"], closure["end_time"], alpha=0.3)
plt.xlabel("Time (seconds)")
plt.ylabel("EAR")
plt.title("EAR and Prolonged Eye Closures")
plt.legend()
plt.grid()
plt.show()

### Channel output (the contract used by fusion)
`channel_output` turns a window of per-frame results into **score, confidence and raw values**. The confidence of the channel is the mean frame confidence over the window, capped by the last half second, so it falls at once when the face or eyes are lost and recovers slowly.

In [11]:
output = channel_output(time_history, eye_state_history, conf_history, window_s=float("inf"))
print(f"score      : {output['score']:.2f}   (0 = alert, 1 = drowsy)")
print(f"confidence : {output['confidence']:.2f}")
for key, value in output["raw"].items():
    print(f"  {key}: {value}")

score      : 0.85   (0 = alert, 1 = drowsy)
confidence : 0.86
  eyes_closed: True
  current_closure_s: 0.8666666666666671
  closure_level: SLOW BLINK
  longest_closure_s: 3.2666666666666675
  recent_longest_closure_s: 3.2666666666666675
  blink_count: 10
  blink_rate_per_min: 27.822861117551586
  perclos: 0.3270868824531516
  face_found_share: 0.9157566302652106
  window_confidence: 0.8636738635642539
  window_s: 21.565


## 3. Step 3: live camera (or a video file, or a stream URL)
`run_video(EarBranch(), source)` is the same code as step 2, now on a rolling 30 s window and with a picture on screen.

* **Calibration:** the first 10 s (eyes open, look at the camera) set the threshold to 0.7 x your median open-eye EAR. No verdict and confidence 0 until then.
* **Big red alert** below the face box for DROWSY (eyes closed 1 s or more) and MICROSLEEP (2 s or more).
* **macOS camera:** the app that runs this notebook (Terminal, VS Code, Jupyter) needs permission under System Settings > Privacy & Security > Camera.
* **Stop:** interrupt the kernel, or set `max_seconds`. In the terminal use `--show window` (press `q` to quit).

In [ ]:
# 0 = laptop camera; try 1 for another camera, or a video path / stream URL
live = run_video(EarBranch(window_s=30, calibrate_s=10), source=0, show="notebook", log_csv="runs/ear_live_log.csv")
print(live["output"])